In [1]:
# Notebook: 10_training_dynamics_ensemble
# Retraining notebook adding the signals reviewers asked for (R1, R5, and the AUM/Data-Maps
# extension of M1). Built on the SAME training loop as nb01 (num_workers=0 for Windows).
# Set N_MODELS=1 for the cheap must-have run (degradation curve + AUM/Data-Maps);
# N_MODELS=3 adds the deep ensemble (R1), ~3x cost.
import os, numpy as np, pandas as pd, torch, seaborn as sns, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
plt.rcParams["axes.edgecolor"]="0.2"; plt.rcParams["axes.linewidth"]=0.8; plt.rcParams["font.family"]="DejaVu Sans"
GREYS=["#111111","#555555","#888888","#bbbbbb","#dddddd"]
DATA_DIR=os.path.join("..","data"); FIG=os.path.join("..","results","figures"); TAB=os.path.join("..","results","tables")
os.makedirs(FIG,exist_ok=True); os.makedirs(TAB,exist_ok=True)

MODEL_NAME="allegro/herbert-base-cased"; MAX_LEN=64
EPOCHS=10; BATCH=128; LR=4e-5; WARMUP_RATIO=0.1; WEIGHT_DECAY=0.01
N_MODELS=3            # 1 = degradation curve + AUM/Data-Maps only; 3 = + ensemble (R1)
HOLDOUT_FRAC=0.10; SUBSET=None; SEED=42; EPS=1e-12
DEVICE="cuda" if torch.cuda.is_available() else "cpu"; assert DEVICE=="cuda","GPU required"
print("device:",DEVICE,"| N_MODELS:",N_MODELS)

df=pd.read_parquet(os.path.join(DATA_DIR,"allenoise_norm.parquet"))
if SUBSET: df=df.sample(SUBSET,random_state=SEED).reset_index(drop=True)
classes=np.sort(df["noisy_category"].unique()); cls2idx={c:i for i,c in enumerate(classes)}; K=len(classes)
df["noisy_id"]=df["noisy_category"].map(cls2idx).astype(int)
df["clean_id"]=df["clean_category"].map(lambda c: cls2idx.get(c,-1)).astype(int)
tr_idx,ho_idx=train_test_split(np.arange(len(df)),test_size=HOLDOUT_FRAC,random_state=SEED,shuffle=True)
print(f"K={K}  train={len(tr_idx):,}  holdout={len(ho_idx):,}")

tok=AutoTokenizer.from_pretrained(MODEL_NAME)
class DS(Dataset):
    def __init__(self,texts,labels=None): self.t=list(texts); self.l=labels
    def __len__(self): return len(self.t)
    def __getitem__(self,i):
        e=tok(self.t[i],truncation=True,max_length=MAX_LEN,padding="max_length",return_tensors="pt")
        it={k:v.squeeze(0) for k,v in e.items()}
        if self.l is not None: it["labels"]=torch.tensor(int(self.l[i]))
        return it
txt=df["text"].values; nid=df["noisy_id"].values; cid=df["clean_id"].values; mis=(nid!=cid).astype(int)
def logits_on(model, rows):
    model.eval(); dl=DataLoader(DS(txt[rows]),batch_size=BATCH,shuffle=False,num_workers=0,pin_memory=True); out=[]
    with torch.no_grad():
        for b in dl:
            b={k:v.to(DEVICE) for k,v in b.items()}
            with torch.cuda.amp.autocast(): lg=model(**b).logits.float()
            out.append(lg.cpu().numpy())
    return np.concatenate(out,0)
def detect_from_logits(lg, rows):
    z=lg-lg.max(1,keepdims=True); P=np.exp(z); P/=P.sum(1,keepdims=True)
    pn=P[np.arange(len(rows)),nid[rows]]; ent=-np.sum(np.clip(P,EPS,1)*np.log(np.clip(P,EPS,1)),1)
    v=cid[rows]>=0; y=mis[rows][v]
    return roc_auc_score(y,(1-pn)[v]), roc_auc_score(y,ent[v]), (P.argmax(1)==cid[rows])[v].mean(), P

aum_sum=np.zeros(len(tr_idx)); conf_sum=np.zeros(len(tr_idx)); conf_sq=np.zeros(len(tr_idx))
deg=[]; ho_prob_sum=np.zeros((len(ho_idx),K),dtype=np.float64)
for m in range(N_MODELS):
    torch.manual_seed(SEED+m); np.random.seed(SEED+m)
    model=AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,num_labels=K).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    dl=DataLoader(DS(txt[tr_idx],nid[tr_idx]),batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True)
    sched=get_linear_schedule_with_warmup(opt,int(WARMUP_RATIO*EPOCHS*len(dl)),EPOCHS*len(dl))
    scaler=torch.cuda.amp.GradScaler()
    for ep in range(EPOCHS):
        model.train()
        for b in tqdm(dl,desc=f"m{m} ep{ep+1}"):
            b={k:v.to(DEVICE) for k,v in b.items()}; opt.zero_grad()
            with torch.cuda.amp.autocast(): loss=model(**b).loss
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        if m==0:
            lg=logits_on(model,tr_idx)
            zt=lg[np.arange(len(tr_idx)),nid[tr_idx]]
            tmp=lg.copy(); tmp[np.arange(len(tr_idx)),nid[tr_idx]]=-1e9
            aum_sum+=zt-tmp.max(1)
            z=lg-lg.max(1,keepdims=True); P=np.exp(z); P/=P.sum(1,keepdims=True)
            pc=P[np.arange(len(tr_idx)),nid[tr_idx]]; conf_sum+=pc; conf_sq+=pc**2
            hl=logits_on(model,ho_idx); a1,ae,acc,_=detect_from_logits(hl,ho_idx)
            deg.append(dict(epoch=ep+1,holdout_clean_acc=round(acc,4),auroc_1mp=round(a1,4),auroc_entropy=round(ae,4)))
            print(f"  [m0 ep{ep+1}] clean_acc={acc:.3f} det_1mp={a1:.3f} det_entropy={ae:.3f}")
    hlg=logits_on(model,ho_idx); z=hlg-hlg.max(1,keepdims=True); Pe=np.exp(z); Pe/=Pe.sum(1,keepdims=True)
    ho_prob_sum+=Pe; del model; torch.cuda.empty_cache()

degdf=pd.DataFrame(deg); degdf.to_csv(os.path.join(TAB,"t10_degradation_curve.csv"),index=False)
print("\n[R5] degradation curve:"); print(degdf.to_string(index=False))
fig,ax=plt.subplots(figsize=(5.2,3.4)); ax2=ax.twinx()
ax.plot(degdf.epoch,degdf.auroc_1mp,marker="o",color=GREYS[0],label="det AUROC (1-p)")
ax.plot(degdf.epoch,degdf.auroc_entropy,marker="s",color=GREYS[1],ls="--",label="det AUROC (entropy)")
ax2.plot(degdf.epoch,degdf.holdout_clean_acc,marker="^",color=GREYS[3],ls=":",label="clean acc")
ax.axhline(0.5,color="black",ls=":",lw=1)
ax.set_xlabel("epoch"); ax.set_ylabel("individual detection AUROC"); ax2.set_ylabel("holdout clean top-1")
ax.legend(frameon=False,fontsize=8,loc="upper right")
for e in ("png","pdf"): fig.savefig(os.path.join(FIG,f"f10_degradation.{e}"),dpi=600,bbox_inches="tight")
plt.close(fig)

aum=aum_sum/EPOCHS; dm_conf=conf_sum/EPOCHS; dm_var=np.sqrt(np.maximum(conf_sq/EPOCHS-dm_conf**2,0))
vt=cid[tr_idx]>=0;

C:\Users\miy\miniconda3\envs\seller_seg\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda | N_MODELS: 3
K=5691  train=452,079  holdout=50,231


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 39808.58it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
cls.sso.sso_relationship.weight            | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arc

  [m0 ep1] clean_acc=0.309 det_1mp=0.556 det_entropy=0.585


m0 ep2:   0%|          | 0/3532 [00:00<?, ?it/s]C:\Users\miy\AppData\Local\Temp\ipykernel_29864\1607456280.py:70: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(): loss=model(**b).loss
m0 ep2: 100%|██████████| 3532/3532 [13:42<00:00,  4.29it/s]
C:\Users\miy\AppData\Local\Temp\ipykernel_29864\1607456280.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(): lg=model(**b).logits.float()


...

  [m0 ep9] clean_acc=0.575 det_1mp=0.513 det_entropy=0.523


m0 ep10:   0%|          | 0/3532 [00:00<?, ?it/s]C:\Users\miy\AppData\Local\Temp\ipykernel_29864\1607456280.py:70: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(): loss=model(**b).loss
m0 ep10: 100%|██████████| 3532/3532 [13:32<00:00,  4.34it/s]
C:\Users\miy\AppData\Local\Temp\ipykernel_29864\1607456280.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(): lg=model(**b).logits.float()


  [m0 ep10] clean_acc=0.577 det_1mp=0.513 det_entropy=0.522


C:\Users\miy\AppData\Local\Temp\ipykernel_29864\1607456280.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(): lg=model(**b).logits.float()
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 49744.71it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED |


[R5] degradation curve:
 epoch  holdout_clean_acc  auroc_1mp  auroc_entropy
     1             0.3092     0.5557         0.5853
     2             0.4412     0.5468         0.5604
     3             0.4908     0.5384         0.5553
     4             0.5207     0.5277         0.5416
     5             0.5402     0.5211         0.5330
     6             0.5563     0.5180         0.5289
     7             0.5632     0.5150         0.5258
     8             0.5709     0.5132         0.5231
     9             0.5747     0.5132         0.5227
    10             0.5770     0.5126         0.5216


In [2]:
# nb10 tail — reprint AUM/Data-Maps detection + ensemble tables (uses in-memory vars from nb10)
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
EPS=1e-12; TAB=os.path.join("..","results","tables"); DATA_DIR=os.path.join("..","data")

# --- [M1 ext] AUM / Data-Maps detectors (train split, vs clean) ---
vt=cid[tr_idx]>=0; yt=mis[tr_idx][vt]
det=pd.DataFrame([
    dict(signal="AUM (low=suspicious)",  auroc=round(roc_auc_score(yt,(-aum)[vt]),3)),
    dict(signal="Data-Maps confidence",  auroc=round(roc_auc_score(yt,(1-dm_conf)[vt]),3)),
    dict(signal="Data-Maps variability", auroc=round(roc_auc_score(yt,dm_var[vt]),3)),
])
det.to_csv(os.path.join(TAB,"t10_aum_datamaps_detection.csv"),index=False)
print("[M1 ext] training-dynamics detectors (train split vs clean):")
print(det.to_string(index=False))

# --- [R1] ensemble vs single (holdout) ---
if 'ho_prob_sum' in globals() and N_MODELS>=2:
    Pens=ho_prob_sum/N_MODELS
    pn=Pens[np.arange(len(ho_idx)),nid[ho_idx]]
    ent=-np.sum(np.clip(Pens,EPS,1)*np.log(np.clip(Pens,EPS,1)),1)
    v=cid[ho_idx]>=0; y=mis[ho_idx][v]
    ens=dict(ensemble_det_1mp=round(roc_auc_score(y,(1-pn)[v]),3),
             ensemble_det_entropy=round(roc_auc_score(y,ent[v]),3),
             ensemble_clean_acc=round((Pens.argmax(1)==cid[ho_idx])[v].mean(),3),
             single_det_1mp=float(degdf.auroc_1mp.iloc[-1]),
             single_clean_acc=float(degdf.holdout_clean_acc.iloc[-1]))
    pd.DataFrame([ens]).to_csv(os.path.join(TAB,"t10_ensemble.csv"),index=False)
    print("\n[R1] ensemble vs single (holdout):")
    for k,vv in ens.items(): print(f"  {k}: {vv}")
else:
    print("\n[R1] ensemble skipped (N_MODELS<2 or ho_prob_sum missing)")

[M1 ext] training-dynamics detectors (train split vs clean):
               signal  auroc
 AUM (low=suspicious)  0.523
 Data-Maps confidence  0.541
Data-Maps variability  0.547

[R1] ensemble vs single (holdout):
  ensemble_det_1mp: 0.513
  ensemble_det_entropy: 0.522
  ensemble_clean_acc: 0.588
  single_det_1mp: 0.5126
  single_clean_acc: 0.577
